In [6]:
import pandas as pd 

In [10]:
df=pd.read_csv("../Sample - Superstore.csv",encoding="latin-1")
df.shape
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [11]:
df.info()
print(df.isna().sum())
print("duplicates:", df.duplicated().sum())
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [12]:
# Standardize column names: "Order Date" -> "order_date", "Sub-Category" -> "sub_category"
df.columns = df.columns.str.strip().str.lower().str.replace(r"[ -]", "_", regex=True)

# Fix data types
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df["postal_code"] = df["postal_code"].astype(str).str.zfill(5)

# Remove exact duplicates
df = df.drop_duplicates()

print(df.shape)
print(df["order_date"].min(), df["order_date"].max())

(9994, 21)
2014-01-03 00:00:00 2017-12-30 00:00:00


In [13]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["profit_margin"] = (df["profit"] / df["sales"]).round(4)
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days

In [16]:
df.shape
df.columns
df.dtypes

row_id                    int64
order_id                    str
order_date       datetime64[us]
ship_date        datetime64[us]
ship_mode                   str
customer_id                 str
customer_name               str
segment                     str
country                     str
city                        str
state                       str
postal_code                 str
region                      str
product_id                  str
category                    str
sub_category                str
product_name                str
sales                   float64
quantity                  int64
discount                float64
profit                  float64
order_year                int32
order_month               int32
profit_margin           float64
ship_days                 int64
dtype: object

In [17]:
%pip install pymysql python-dotenv sqlalchemy


   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [pymysql]
   -------------------- ------------------- 1/2 [pymysql]
   -------------------- ------------------- 1/2 [pymysql]
   -------------------- ------------------- 1/2 [pymysql]
   -------------------- ------------------- 1/2 [pymysql]
   ---------------------------------------- 2/2 [pymysql]

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
from dotenv import load_dotenv, find_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

load_dotenv(find_dotenv())
pw = quote_plus(os.environ["DB_PASSWORD"])
url = f"mysql+pymysql://{os.environ['DB_USER']}:{pw}@{os.environ['DB_HOST']}:3306/retail"
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT VERSION()")).fetchone())

('8.0.44',)


In [22]:
df.to_sql("order_raw",engine,index=False,if_exists="replace", chunksize=1000)
print("loaded",len(df),"rows")

loaded 9994 rows
